In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

### Import Data

In [ ]:
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered.csv'))
df.info()

In [ ]:
df2 = df[['수종명', '흉고직경', '수고', '수령', '지하고']]
cm_to_inch = 0.3937
cm_to_ft = 0.0328084
df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)

In [ ]:
df2.columns = ['Species', 'DBH(inch)', 'H(ft)', 'A', 'CBH(ft)']
df2.info

In [ ]:
# 수관비율 생성
df2['CR'] = 1 - (df2['CBH(ft)'] / df2['H(ft)'])

In [ ]:
df2.head(10)

In [ ]:
def drawPairPlot(df, s_name, opt_save, f_name=None):
    condition = (df['Species'] == s_name)
    df_species = df.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CBH(ft)', 'CR']]
    sns.pairplot(df_species, kind='reg', plot_kws={'line_kws' : {'color' : 'orange'}})
    if opt_save: 
        save_dir = r'D:/ForestFire/CBH/fig'
        plt.savefig(os.path.join(save_dir, f_name))

In [ ]:
df2.groupby('Species').count().sort_values(by=['DBH(inch)'], ascending=False).iloc[:20, :]

In [ ]:
# 소나무 scatterplot
drawPairPlot(df2, '소나무', opt_save=True, f_name='소나무-Scatter.png')

In [ ]:
# 굴참나무 scatterplot
drawPairPlot(df2, '굴참나무', opt_save=True, f_name='굴참-Scatter.png')

In [ ]:
# 왕벚나무 scatterplot
drawPairPlot(df2, '왕벚나무', opt_save=True, f_name='왕벚-Scatter.png')

In [ ]:
# 고로쇠 scatterplot
drawPairPlot(df2, '고로쇠나무', opt_save=True, f_name='고로쇠-Scatter(30).png')

### Build species-dependent Model

In [ ]:
from scipy.optimize import minimize

In [ ]:
# 임상도 수종 및 코드 분류에 따라 수조ㅜㅇ코드(SID) 부여하기
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 10, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 30, 61, 62, 63, 64]
name_lst = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '기타침엽수', '상수리나무', '신갈나무', '굴참나무', '기타 참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무',
           '호두나무', '백합나무', '포플러', ' 벚나무', '느티나무', '층층나무', '아까시나무', '기타활엽수', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무']
print(len(id_lst), len(name_lst))
s_dict = {j : i for i, j in zip(id_lst, name_lst)}
sid_lst = []
for idx, name in enumerate(df2['Species']):
    if name in s_dict.keys(): sid_lst.append(s_dict[name])
    else:
        if (df.loc[idx, '침활구분'] == '활엽수'): sid_lst.append(30)
        elif (df.loc[idx, '침활구분'] == '침엽수'): sid_lst.append(10)
len(sid_lst) == df2.shape[0]

In [ ]:
# Dataframe에 SID 추가하기
df2.insert(0, 'SID', sid_lst)

In [ ]:
df2

In [ ]:
# record array 만들기: (species_id, data_count, a, b, r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)
rec = np.zeros((len(name_lst), 10))
rec[:, 0] = id_lst
print(rec.shape)
rec

In [ ]:
# 함수 정의
def func(X, a, b):
    dbh, h, age = X # Unpacking the tuple
    return 1 - np.exp(-(a + b * (1/age)) * dbh/h)

def loss_func(params, X, y):
    y_pred = func(X, *params)
    return np.sum((y - y_pred) ** 2) + 0.1 * np.sum(params**2) # L2 규제 적용

In [ ]:
for sid in tqdm(df2['SID'].unique()):
    condition = (df2['SID'] == sid)
    df_species = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CR']]
    cnt = len(df_species)
    popt = [-999, -999]
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    
    if cnt > 2:
        # train-test data split
        X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :3], df_species['CR'], test_size=.2, random_state=44)
        
        # create train-test X-array for curve-fit
        X_train = np.array(X_train).T
        X_test = np.array(X_test).T
    
        # train curvefit
        popt, pcov = curve_fit(func, np.array(X_train), np.array(y_train))
        y_pred = func(X_train, *popt)
        r2_train = r2_score(y_train, y_pred)
        mae_train = mean_absolute_error(y_train, y_pred)
        rmse_train = root_mean_squared_error(y_train, y_pred)
        
        # test curvefit
        y_pred2 = func(X_test, *popt)
        r2_test = r2_score(y_test, y_pred2)
        mae_test = mean_absolute_error(y_test, y_pred2)
        rmse_test = root_mean_squared_error(y_test, y_pred2)

    # save the result in the record list
    rec[rec[:, 0] == sid, :] = [sid, cnt] + list(popt) + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test]
    result_dir = r'D:/ForestFire/CBH/result'
    np.savetxt(os.path.join(result_dir, 'CR_Dyer_result.txt'), rec, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'Par1', 'Par2', 'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

In [ ]:
# 소나무 pcov 분석
sid = '왕벚나무'
condition = (df2['Species'] == sid)
df_species = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CR']]

# train-test data split
X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :3], df_species['CR'], test_size=.2) #, random_state=44)

# create train-test X-array for curve-fit
X_train = np.array(X_train).T
X_test = np.array(X_test).T

# train curvefit
popt, pcov = curve_fit(func, np.array(X_train), np.array(y_train))
result_reg = minimize(loss_func, x0=popt, args=(X_train, y_train))
best_params = result_reg.x
y_pred = func(X_train, *popt)
r2_train = r2_score(y_train, y_pred)
mae_train = mean_absolute_error(y_train, y_pred)
rmse_train = root_mean_squared_error(y_train, y_pred)

# test curvefit
y_pred2 = func(X_test, *popt)
r2_test = r2_score(y_test, y_pred2)
mae_test = mean_absolute_error(y_test, y_pred2)
rmse_test = root_mean_squared_error(y_test, y_pred2)

In [ ]:
print(r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)

In [ ]:
pcov

In [ ]:
df_result = pd.DataFrame(rec, columns=['SID','Count' ,'Par1', 'Par2', 'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'])
df_result.insert(1, 'SName', name_lst)
df_result.to_csv(os.path.join(result_dir, 'CR_Dyer_result.csv'), encoding='cp949')
df_result2 = df_result.loc[(df_result['Count'] > 30), :].sort_values(by='Count', ascending=False).reset_index(drop=True)

In [ ]:
df_result = pd.DataFrame(rec, columns=['SID','Count' ,'Par1', 'Par2', 'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'])
df_result.insert(1, 'SName', name_lst)
df_result.to_csv(os.path.join(result_dir, 'CR_Dyer_result.csv'), encoding='cp949')
df_result2 = df_result.loc[(df_result['Count'] > 30), :].sort_values(by='Count', ascending=False).reset_index(drop=True)

import matplotlib.font_manager as fm

# Load the dataset

# Set font to support Korean characters
plt.rc('font', family='Malgun Gothic')  # Use 'Malgun Gothic' for Windows or 'AppleGothic' for Mac
plt.rcParams['axes.unicode_minus'] = False  # Ensure minus signs are displayed correctly

# Set figure size
plt.figure(figsize=(12, 6))

# Define bar width and positions
bar_width = 0.4
x = np.arange(len(df_result2["SName"]))

# Create the bars
plt.bar(x - bar_width/2, df_result2["r2_tr"], width=bar_width, label="r2_tr", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, df_result2["r2_te"], width=bar_width, label="r2_te", color="orange", alpha=0.7)

# Add count annotations
for i in range(len(df_result2)):
    plt.text(x[i] - bar_width/2, df_result2["r2_tr"][i] + 0.02, f'{df_result2["r2_tr"][i]:.2f}', ha='center', fontsize=10)
    plt.text(x[i] + bar_width/2, df_result2["r2_te"][i] + 0.02, f'{df_result2["r2_te"][i]:.2f}', ha='center', fontsize=10)

# Labels and title
plt.xlabel("SName", fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title("R2 of the equations by species", fontsize=14)
plt.xticks(ticks=x, labels=df_result2["SName"], rotation=45, ha="right")
plt.legend()

# Show plot
plt.tight_layout()
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, 'R2 by species.png'))
plt.show()


### 임상별 수관비율 추정식

In [ ]:
imid_lst = []
for name in df['침활구분']:
    if (name == '활엽수'): imid_lst.append(30)
    elif (name == '침엽수'): imid_lst.append(10)
len(imid_lst) == df2.shape[0]
df2['MID'] = imid_lst

In [ ]:
df2.loc[df2['MID'] == 10]

In [ ]:
# record array 만들기: (species_id, data_count, a, b, r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)
rec2 = np.zeros((len(df['침활구분'].unique()), 10))
rec2[:, 0] = [10, 30]
print(rec2.shape)
rec2

In [ ]:
for mid in tqdm(df2['MID'].unique()):
    condition = (df2['MID'] == mid)
    df_species = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CR']]
    cnt = len(df_species)
    popt = [-999, -999]
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    
    if cnt > 2:
        # train-test data split
        X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :3], df_species['CR'], test_size=.3)
        
        # create train-test X-array for curve-fit
        X_train = np.array(X_train).T
        X_test = np.array(X_test).T
    
        # train curvefit
        popt, pcov = curve_fit(func, np.array(X_train), np.array(y_train))
        y_pred = func(X_train, *popt)
        r2_train = r2_score(y_train, y_pred)
        mae_train = mean_absolute_error(y_train, y_pred)
        rmse_train = root_mean_squared_error(y_train, y_pred)
        
        # test curvefit
        y_pred2 = func(X_test, *popt)
        r2_test = r2_score(y_test, y_pred2)
        mae_test = mean_absolute_error(y_test, y_pred2)
        rmse_test = root_mean_squared_error(y_test, y_pred2)

    # save the result in the record list
    rec2[rec2[:, 0] == mid, :] = [mid, cnt] + list(popt) + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test]
    result_dir = r'D:/ForestFire/CBH/result'
    np.savetxt(os.path.join(result_dir, 'CR_Dyer_Imsang_result.txt'), rec2, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'Par1', 'Par2', 'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

In [ ]:
df_result = pd.DataFrame(rec2, columns=['SID','Count' ,'Par1', 'Par2', 'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'])
df_result.insert(1, 'MName', ['침엽수', '활엽수'])
df_result.to_csv(os.path.join(result_dir, 'CR_Dyer_Imsang_result.csv'), encoding='cp949')
df_result2 = df_result.sort_values(by='Count', ascending=False).reset_index(drop=True)

# Set figure size
plt.figure(figsize=(5, 7))

# Define bar width and positions
bar_width = 0.4
x = np.arange(len(df_result2["MName"]))

# Create the bars
plt.bar(x - bar_width/2, df_result2["r2_tr"], width=bar_width, label="r2_tr", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, df_result2["r2_te"], width=bar_width, label="r2_te", color="orange", alpha=0.7)

# Add count annotations
for i in range(len(df_result2)):
    plt.text(x[i] - bar_width/2, df_result2["r2_tr"][i] + 0.002, f'{df_result2["r2_tr"][i]:.2f}', ha='center', fontsize=10)
    plt.text(x[i] + bar_width/2, df_result2["r2_te"][i] + 0.002, f'{df_result2["r2_te"][i]:.2f}', ha='center', fontsize=10)

# Labels and title
plt.xlabel("MName", fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title("R2 of the equations by imsang", fontsize=14)
plt.xticks(ticks=x, labels=df_result2["MName"], rotation=45, ha="right")
plt.legend()

# Show plot
plt.tight_layout()
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, 'R2 by Imsang.png'))
plt.show()

In [ ]:
# 침엽수
condition = (df2['MID'] == 10)
df_coni = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CBH(ft)','CR']]
sns.pairplot(df_coni, kind='reg', plot_kws={'line_kws':{'color':'orange'}})
plt.suptitle("Scatterplot of Coniferous Trees", fontsize=20, y=1.02)
plt.savefig(os.path.join(fig_dir, 'Coniferous-Scatter.png'))
plt.show()

In [ ]:
# 활엽수
condition = (df2['MID'] == 30)
df_deci = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CBH(ft)','CR']]
sns.pairplot(df_deci, kind='reg', plot_kws={'line_kws':{'color':'orange'}})
plt.suptitle("Scatterplot of Deciduous Trees", fontsize=20, y=1.02)
plt.savefig(os.path.join(fig_dir, 'Deciduous-Scatter.png'))

### 지하고 예측

In [ ]:
# predicting 지하고
def func2(X, a, b):
    dbh, h, age = X # Unpacking the tuple
    return h * np.exp(-(a + b * (1/age)) * (dbh / h))

In [ ]:
# extract data by species
s_name = '소나무'
condition = (df2['Species'] == s_name)
df_species = df2.loc[condition, ['DBH(inch)', 'H(ft)', 'A', 'CBH']]
X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :3], df_species['CBH(ft)'], test_size=.1)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
X_train = np.array(X_train).T
X_test = np.array(X_test).T
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# train curvefit with func2
popt, pcov = curve_fit(func, np.array(X_train), np.array(y_train))
print(popt)
y_pred = func(X_train, *popt)
print(r2_score(y_train, y_pred), mean_absolute_error(y_train, y_pred), mean_squared_error(y_train, y_pred))